# Dataset Large + Mid — Inventaris & Kualitas Data

**Tujuan**: menyiapkan universe tanpa *survivorship bias* untuk mengulang riset 07/08. Sebelumnya hanya
10 coin CSV (dipilih karena **sekarang** sudah besar). Sekarang didownload **semua simbol Large & Mid**
di `flex_params` (50 simbol, aktif maupun nonaktif) dari Binance Vision, format & rentang sama dengan
data lama: candle harian, 2021-01 → 2026-06.

Cara download (dari folder `notebooks/`):
```
python -m data_source.download_binance_vision <SIMBOL> --start 2021-01 --end 2026-06
```
Catatan: **GRAMUSDT** (Gram, dulu Toncoin) tidak ada di arsip dengan nama itu — didownload sebagai
`TONUSDT` lalu disimpan sebagai `GRAMUSDT_1d_full.csv` (aset yang sama).

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT)); sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from data_source.loader import load_symbol, discover_symbols
from src.databases import get_connection
MIN_VOLATILITY = 0.012   # median ATR% < 1,2% = stablecoin (dulu dari service rekomendasi V5)

def atr_pct(high, low, close, window=14):
    """ATR(14) sebagai % harga (dulu src/strategy/indicators/atr.py)."""
    prev = close.shift(1)
    tr = pd.concat([high - low, (high - prev).abs(), (low - prev).abs()], axis=1).max(axis=1)
    return tr.rolling(window, min_periods=window).mean() / close

pd.set_option("display.width", 180); pd.set_option("display.max_rows", 100)

with get_connection() as conn:
    meta = pd.DataFrame(conn.execute(
        "SELECT value_param, description, is_active FROM flex_params "
        "WHERE type_param='SIMBOL_CRYPTO' AND deleted_at IS NULL AND description IN ('Large','Mid')"
    ).fetchall(), columns=["symbol", "kategori", "aktif"]).set_index("symbol")
available = set(discover_symbols("1d", "full"))
print(f"Large/Mid di flex_params: {len(meta)} | CSV tersedia: {len(meta.index.intersection(list(available)))}")
print("Belum ada CSV:", sorted(set(meta.index) - available) or "-")

Large/Mid di flex_params: 50 | CSV tersedia: 50
Belum ada CSV: -


In [2]:
END = pd.Timestamp("2026-06-30")
rows = []
for s in meta.index:
    if s not in available:
        continue
    d = load_symbol(s, "1d", "full")
    full = pd.date_range(d.index.min(), END, freq="D")
    missing = full.difference(d.index)
    gaps = pd.Series(missing).diff().dt.days.ne(1).cumsum().value_counts().max() if len(missing) else 0
    a = atr_pct(d["high"], d["low"], d["close"]).median()
    rows.append(dict(symbol=s, kategori=meta.at[s, "kategori"], aktif=meta.at[s, "aktif"],
                     mulai=d.index.min().date(), akhir=d.index.max().date(), hari=len(d),
                     hari_hilang=len(missing), gap_terpanjang=int(gaps),
                     atr_median=a, stablecoin=a < MIN_VOLATILITY,
                     histori_sejak_2021=d.index.min() <= pd.Timestamp("2021-01-31")))
inv = pd.DataFrame(rows).sort_values(["kategori", "mulai", "symbol"]).set_index("symbol")
inv.assign(atr_median=inv["atr_median"].map(lambda v: f"{v:.2%}"))

,kategori,aktif,mulai,akhir,hari,hari_hilang,gap_terpanjang,atr_median,stablecoin,histori_sejak_2021
symbol,,,,,,,,,,
BTCUSDT,Large,True,2018-01-01,2026-06-30,3103,0,0,4.27%,False,True
BNBUSDT,Large,True,2021-01-01,2026-06-30,2007,0,0,4.53%,False,True
DOGEUSDT,Large,True,2021-01-01,2026-06-30,2007,0,0,7.10%,False,True
ETHUSDT,Large,True,2021-01-01,2026-06-30,2007,0,0,5.25%,False,True
LINKUSDT,Large,True,2021-01-01,2026-06-30,2007,0,0,6.97%,False,True
SOLUSDT,Large,True,2021-01-01,2026-06-30,2007,0,0,7.44%,False,True
TRXUSDT,Large,True,2021-01-01,2026-06-30,2007,0,0,3.21%,False,True
USDCUSDT,Large,True,2021-01-01,2022-09-26,634,1373,1373,0.06%,True,True
XRPUSDT,Large,True,2021-01-01,2026-06-30,2007,0,0,5.93%,False,True


In [3]:
usable = inv[~inv["stablecoin"]]
print(f"Total CSV Large/Mid           : {len(inv)}")
print(f"Stablecoin / aset tidak volatil: {inv['stablecoin'].sum()} -> {sorted(inv.index[inv['stablecoin']])}")
print(f"Coin bisa dipakai strategi     : {len(usable)} (aktif {int(usable['aktif'].sum())})")
print(f"  punya histori sejak 2021     : {int(usable['histori_sejak_2021'].sum())}")
print(f"  listing belakangan (> 2021-01): {int((~usable['histori_sejak_2021']).sum())} -> "
      f"{sorted(usable.index[~usable['histori_sejak_2021']])}")
print(f"Data bolong (hari hilang > 0)  : {sorted(inv.index[inv['hari_hilang'] > 0])}")
lst = usable.groupby(pd.to_datetime(usable["mulai"]).dt.year).size()
print("\nJumlah coin (non-stablecoin) per tahun mulai data:", lst.to_dict())

Total CSV Large/Mid           : 50
Stablecoin / aset tidak volatil: 7 -> ['BFUSDUSDT', 'RLUSDUSDT', 'USD1USDT', 'USDCUSDT', 'USDEUSDT', 'USDSUSDT', 'UUSDT']
Coin bisa dipakai strategi     : 43 (aktif 22)
  punya histori sejak 2021     : 22
  listing belakangan (> 2021-01): 21 -> ['ARBUSDT', 'ASTERUSDT', 'ENAUSDT', 'GRAMUSDT', 'ICPUSDT', 'JUPUSDT', 'MORPHOUSDT', 'ONDOUSDT', 'PEPEUSDT', 'POLUSDT', 'PUMPUSDT', 'QNTUSDT', 'RENDERUSDT', 'SHIBUSDT', 'SKYUSDT', 'SUIUSDT', 'TAOUSDT', 'WBETHUSDT', 'WLDUSDT', 'WLFIUSDT', 'XAUTUSDT']
Data bolong (hari hilang > 0)  : ['USDCUSDT']

Jumlah coin (non-stablecoin) per tahun mulai data: {2018: 1, 2021: 24, 2023: 5, 2024: 6, 2025: 6, 2026: 1}


## Kesimpulan

**Dataset lengkap: 50/50 simbol Large+Mid** punya CSV (10 lama + 39 baru + GRAM via TONUSDT).
Kualitas data bagus: tidak ada hari bolong kecuali **USDCUSDT** (pair berhenti diperdagangkan 2022-09).

| Kelompok | Jumlah | Catatan |
|---|---|---|
| Stablecoin (median ATR < 1.2%) | 7 | USDC, USD1, BFUSD, USDE, U, RLUSD, USDS — **dibuang** otomatis oleh filter strategi |
| Coin dengan histori sejak 2021 | 22 | inti untuk periode latih 2021–2023 |
| Coin listing belakangan | 21 | 2021: SHIB, ICP, QNT · 2023: ARB, SUI, PEPE, WLD, WBETH · 2024: JUP, ENA, TAO, RENDER, GRAM, POL · 2025: ONDO, WLFI, PUMP, SKY, MORPHO, ASTER · 2026: XAUT |

### Hal yang harus diperhatikan saat riset ulang (notebook berikutnya)
1. **WBETH ≈ ETH** (ETH yang di-stake, harganya mengikuti ETH) — sebaiknya dikeluarkan supaya
   momentum ETH tidak terhitung dua kali.
2. **PAXG & XAUT = token emas**, bukan crypto murni (ATR 1.3–1.7%, lolos filter stablecoin).
   Di validasi 07, PAXG sering dipegang saat crypto lemah. Perlu diputuskan: boleh dipegang
   (jadi "safe haven") atau dikeluarkan.
3. **Survivorship bias berkurang, tapi belum hilang.** Universe ini tetap "coin yang Large/Mid
   **hari ini**". Coin yang dulu besar lalu jatuh/delisting (mis. LUNA, FTT) tidak ikut. Tapi jauh lebih
   baik dari 10 coin sebelumnya: 43 coin, termasuk yang performanya buruk sejak 2021
   (mis. ALGO, ICP, ETC, DOT, NEAR).
4. Coin baru otomatis ikut strategi sejak cukup histori (MA50 butuh 50 hari), jadi periode uji
   2024–2026 punya lebih banyak pilihan daripada periode latih — realistis, seperti kondisi nyata.